# 04. 온체인 공급량 데이터 수집 (DefiLlama)

**출처**: DefiLlama 무료 API (`stablecoins.llama.fi`)  
**저장 위치**: `data/raw/onchain_supply.csv`

## 수집 배경

원래 목표: Exchange Flow (거래소 유입/유출) + Whale Movement (고래 이동)  
→ Glassnode, CryptoQuant 등 **모두 유료** (월 $29~)  
→ **DefiLlama 공급량 변화를 대리변수로 활용**

## 대리변수 논리

- `supply_change > 0` (발행, Mint): 대형 기관/고래가 USD를 맡기고 스테이블코인 발행 요청 → 매수 압력
- `supply_change < 0` (소각, Burn): 스테이블코인을 반납하고 USD 회수 → 매도/탈출 신호
- 이 변화가 급격할수록 대형 거래 발생 가능성 높음

## DefiLlama 코인 ID
| 코인 | ID |
|------|----|
| USDT | 1 |
| USDC | 2 |
| DAI  | 5 |

In [ ]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timezone

SAVE_DIR   = os.path.join('..', 'data', 'raw')
START_DATE = '2019-11-22'
END_DATE   = datetime.today().strftime('%Y-%m-%d')

# DefiLlama 스테이블코인 ID
COINS = {
    'USDT': '1',
    'USDC': '2',
    'DAI':  '5'
}

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'application/json'
}

In [ ]:
def fetch_supply_history(coin_id, chain='all'):
    """
    DefiLlama 스테이블코인 역사 공급량 수집
    - chain='all': 모든 체인 합산
    - chain='Ethereum': 이더리움 체인만
    - chain='Tron': 트론 체인만 (USDT 주요 체인)
    """
    if chain == 'all':
        url = f'https://stablecoins.llama.fi/stablecoincharts/all?stablecoin={coin_id}'
    else:
        url = f'https://stablecoins.llama.fi/stablecoincharts/{chain}?stablecoin={coin_id}'

    res = requests.get(url, headers=HEADERS, timeout=30)
    res.raise_for_status()
    return res.json()


def unix_to_date(ts):
    """Unix 타임스탬프 → YYYY-MM-DD 문자열 변환"""
    return datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime('%Y-%m-%d')

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
all_dfs = []

for symbol, coin_id in COINS.items():
    print(f'\n수집 중: {symbol} (id={coin_id})')

    # ── 1. 전체 체인 합산 공급량 ──────────────────────────────────────
    # totalCirculating: 현재 유통 중인 총 공급량 (USD 가치)
    # totalUnreleased:  발행사(Tether/Circle)가 보유 중인 미유통 물량
    try:
        data_all = fetch_supply_history(coin_id, chain='all')
        rows = []
        for d in data_all:
            date_str = unix_to_date(d['date'])
            if date_str < START_DATE:
                continue
            rows.append({
                'Date':                 date_str,
                f'circ_{symbol}':       d.get('totalCirculating', {}).get('peggedUSD'),
                f'unreleased_{symbol}': d.get('totalUnreleased', {}).get('peggedUSD'),
                f'minted_{symbol}':     d.get('totalMintedUSD', {}).get('peggedUSD'),
            })
        df_all = pd.DataFrame(rows).drop_duplicates('Date').sort_values('Date').reset_index(drop=True)

        # 전일 대비 유통량 변화 계산 (양수=발행, 음수=소각)
        df_all[f'supply_change_{symbol}'] = df_all[f'circ_{symbol}'].diff()
        print(f'  전체 체인: {len(df_all)}개')
        time.sleep(1)
    except Exception as e:
        print(f'  오류 (전체): {e}')
        df_all = pd.DataFrame(columns=['Date'])

    # ── 2. 이더리움 체인 유통량 ─────────────────────────────────────
    # USDT: 이더리움 체인이 두 번째로 큰 체인 (첫 번째는 Tron)
    # USDC: 이더리움 체인이 주 체인
    # DAI:  이더리움이 거의 유일한 체인 (MakerDAO 기반)
    try:
        data_eth = fetch_supply_history(coin_id, chain='Ethereum')
        rows_eth = []
        for d in data_eth:
            date_str = unix_to_date(d['date'])
            if date_str < START_DATE:
                continue
            rows_eth.append({
                'Date':                   date_str,
                f'eth_circ_{symbol}':     d.get('totalCirculating', {}).get('peggedUSD'),
                f'eth_minted_{symbol}':   d.get('totalMintedUSD', {}).get('peggedUSD'),
            })
        df_eth = pd.DataFrame(rows_eth).drop_duplicates('Date').sort_values('Date').reset_index(drop=True)
        df_eth[f'eth_supply_change_{symbol}'] = df_eth[f'eth_circ_{symbol}'].diff()
        print(f'  Ethereum 체인: {len(df_eth)}개')
        time.sleep(1)
    except Exception as e:
        print(f'  오류 (Ethereum): {e}')
        df_eth = pd.DataFrame(columns=['Date'])

    # ── 3. Tron 체인 (USDT 전용) ────────────────────────────────────
    # USDT는 Tron 체인이 압도적 1위 → Tron 유통량 변화가 중요한 신호
    if symbol == 'USDT':
        try:
            data_tron = fetch_supply_history(coin_id, chain='Tron')
            rows_tron = []
            for d in data_tron:
                date_str = unix_to_date(d['date'])
                if date_str < START_DATE:
                    continue
                rows_tron.append({
                    'Date':              date_str,
                    f'tron_circ_{symbol}': d.get('totalCirculating', {}).get('peggedUSD'),
                })
            df_tron = pd.DataFrame(rows_tron).drop_duplicates('Date').sort_values('Date').reset_index(drop=True)
            df_tron[f'tron_supply_change_{symbol}'] = df_tron[f'tron_circ_{symbol}'].diff()
            print(f'  Tron 체인: {len(df_tron)}개')
            time.sleep(1)
        except Exception as e:
            print(f'  오류 (Tron): {e}')
            df_tron = pd.DataFrame(columns=['Date'])
    else:
        df_tron = pd.DataFrame(columns=['Date'])

    # 코인 내 3개 체인 병합
    df_coin = df_all
    if not df_eth.empty and 'Date' in df_eth.columns:
        df_coin = df_coin.merge(df_eth, on='Date', how='left')
    if symbol == 'USDT' and not df_tron.empty and 'Date' in df_tron.columns:
        df_coin = df_coin.merge(df_tron, on='Date', how='left')

    all_dfs.append(df_coin)
    print(f'  완료: {len(df_coin)}행 × {len(df_coin.columns)}열')
    time.sleep(2)

In [ ]:
# 3개 코인 데이터 병합 후 저장
merged = all_dfs[0]
for df in all_dfs[1:]:
    merged = merged.merge(df, on='Date', how='outer')
merged = merged.sort_values('Date').reset_index(drop=True)
merged = merged[merged['Date'] >= START_DATE].reset_index(drop=True)

save_path = os.path.join(SAVE_DIR, 'onchain_supply.csv')
merged.to_csv(save_path, index=False)
print(f'저장 완료: {save_path}')
print(f'총 {len(merged)}행 × {len(merged.columns)}열')
print('컬럼:', list(merged.columns))
merged.head(3)